In [ ]:
#!/usr/bin/env python3
import torch
from torch import nn

In [6]:
ANGLE_CLASSES = list(range(-90, 91, 15))
NUM_ANGLE_CLASSES = len(ANGLE_CLASSES)

def angle_to_class(angle_deg):
    angle_deg = float(angle_deg)
    closest_idx = min(range(len(ANGLE_CLASSES)), key=lambda i: abs(ANGLE_CLASSES[i] - angle_deg))
    return closest_idx

def class_to_angle(class_idx):
    return ANGLE_CLASSES[int(class_idx)]

print(angle_to_class(90))
print(angle_to_class(80))
print(class_to_angle(0))
print(class_to_angle(6))

12
11
-90
0


In [11]:
#!/usr/bin/env python3
import torch
from torch import nn

class SpeakerDirectionCNN(nn.Module):
    def __init__(self, 
                 n_mels=64,
                 n_angle_classes=NUM_ANGLE_CLASSES,
                 size_ch1=16,
                 size_ch2=32,
                 size_ch3=64,
                 size_kernel=3):
        super().__init__()

        self.n_angle_classes = n_angle_classes

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, size_ch1, size_kernel, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(size_ch1, size_ch2, size_kernel, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(size_ch2, size_ch3, size_kernel, padding='same'),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(size_ch3, n_angle_classes)

    def forward(self, x):
        x = self.conv_layers(x)        # [B, size_ch3, 1, 1]
        x = x.view(x.size(0), -1)      # [B, size_ch3]
        x = self.fc(x)                 # [B, 13]
        return x

model = SpeakerDirectionCNN(n_mels=64, n_angle_classes=NUM_ANGLE_CLASSES)
x = torch.randn(2, 1, 64, 160)   # T=160 に縮めて試す（重くならない）
y = model(x)
print("input:", x.shape)
print("output:", y.shape)
print("pred class:", y[0].argmax().item(), "angle:", class_to_angle(y[0].argmax().item()))

input: torch.Size([2, 1, 64, 160])
output: torch.Size([2, 13])
pred class: 1 angle: -75
